[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/openquantumhardware/qick/blob/merge-hrl-main/emulator/notebooks/tutorial/03_Advanced_Timing_emu.ipynb)

# 03 - Advanced Timing with tProc v2 (Emulator Edition)

**Objective:** Understand the critical differences between `delay()`, `wait()`, `delay_auto()`, and `wait_auto()`. Learn how to manage the timeline correctly to avoid pulse collisions and timing errors.

This is a 1:1 port of the on-hardware tutorial `03_Advanced_Timing.ipynb` (mirrored under [`../../docs/source/tutorials/`](../../docs/source/tutorials/) in this repo). Only Section 1 (connecting to the board) is adapted -- every other cell in this notebook only inspects the *compiled program* (`prog.asm()`), which is pure Python compilation against `soccfg` and never touches `soc` at runtime. So none of these cells actually run a Verilator simulation; they're identical on hardware and emulator.

## 0. Colab Setup

Skip this cell if you're running locally with the `qick-venv` kernel (see [`emulator/README.md`](https://github.com/openquantumhardware/qick/blob/merge-hrl-main/emulator/README.md)) -- it's a no-op there. In Google Colab it clones this repo, builds Verilator 5.042 from source, and installs the Python dependencies. **First run takes a few minutes** (compiling Verilator); re-running later cells, or reconnecting to the same runtime, reuses what's already built.

In [ ]:
import os
import sys
import pathlib
import subprocess

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


def run(cmd, **kwargs):
    """Run a shell command, streaming output, and raise loudly on failure.

    Using this instead of bare `!command` / get_ipython().system() so a
    failure here stops the bootstrap immediately with a real traceback,
    instead of silently letting later cells fail with a confusing
    ModuleNotFoundError.
    """
    print(f"+ {cmd}")
    result = subprocess.run(cmd, shell=True, **kwargs)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed (rc={result.returncode}): {cmd}")


if IN_COLAB:
    REPO_URL = "https://github.com/openquantumhardware/qick.git"
    REPO_BRANCH = "merge-hrl-main"
    REPO_DIR = pathlib.Path("/content/qick")

    if not REPO_DIR.exists():
        print(f"Cloning {REPO_URL} @ {REPO_BRANCH} ...")
        run(f"git clone --branch {REPO_BRANCH} --depth 1 {REPO_URL} {REPO_DIR}")
    os.chdir(REPO_DIR)

    print("Fetching git submodules (Verilator source, AXI/common_cells) ...")
    run("git submodule update --init --recursive --depth 1")

    # These are needed unconditionally, whether Verilator itself ends up
    # prebuilt or compiled from source: `make verilate` always invokes g++
    # (via ccache) to compile the *generated* C++ model for each testbench
    # build, every session. Skipping this on the prebuilt-Verilator path
    # broke `make verilate` with "ccache: No such file or directory".
    print("Installing build/runtime dependencies (apt) ...")
    run("DEBIAN_FRONTEND=noninteractive apt-get -qq update")
    run(
        "DEBIAN_FRONTEND=noninteractive apt-get -qq install -y "
        "git help2man perl python3 make autoconf g++ flex bison ccache "
        "libgoogle-perftools-dev numactl perl-doc libfl-dev zlib1g-dev"
    )

    def verilator_version():
        try:
            r = subprocess.run(["verilator", "--version"], capture_output=True, text=True)
        except FileNotFoundError:
            return ""
        return r.stdout

    if "5.042" not in verilator_version():
        # Fast path: a prebuilt Verilator 5.042, compiled on a real Colab
        # runtime and hosted as a GitHub Release asset -- avoids a ~10-15
        # minute source compile on every session. Falls back to building
        # from source if the download/extraction doesn't work out (e.g. the
        # asset is gone, or a future Colab image isn't binary-compatible).
        PREBUILT_URL = (
            "https://github.com/openquantumhardware/qick/releases/download/"
            "verilator-5.042-colab/verilator-5.042-colab-x86_64.tar.gz"
        )
        prebuilt_ok = False
        print("Trying prebuilt Verilator 5.042 (fast path) ...")
        try:
            run(f"curl -fL -o /tmp/verilator-prebuilt.tar.gz {PREBUILT_URL}")
            run("tar -C /usr/local -xzf /tmp/verilator-prebuilt.tar.gz")
            prebuilt_ok = "5.042" in verilator_version()
        except RuntimeError as e:
            print(f"Prebuilt Verilator download/install failed ({e}); building from source instead.")

        if prebuilt_ok:
            print("Prebuilt Verilator 5.042 installed.")
        else:
            print("Building Verilator 5.042 from source (a few minutes, first run only) ...")
            os.chdir(REPO_DIR / "emulator" / "submodules" / "verilator")
            run("autoconf")
            run("./configure")
            run("make -j$(nproc)")
            run("make install")
            os.chdir(REPO_DIR)
            final_version = verilator_version()
            if "5.042" not in final_version:
                raise RuntimeError(
                    "Verilator build finished but 'verilator --version' still doesn't "
                    f"report 5.042 (got: {final_version!r})"
                )
            print(final_version)
    else:
        print("Verilator 5.042 already installed, skipping build.")

    # Scientific-Python deps only -- NOT jupyter/ipykernel, which would fight
    # with Colab's own kernel runtime. `emulator/requirements.txt` is for the
    # local venv+Jupyter setup (see emulator/README.md); Colab needs a subset.
    print("Installing Python dependencies ...")
    run("pip install numpy scipy matplotlib tqdm")
    run("pip install -e .")

    # Belt-and-suspenders: insert the source dirs directly, so `import qick`
    # works below even if pip's editable-install .pth file isn't picked up by
    # this already-running interpreter (a known Colab quirk).
    for rel in ("qick_lib", "emulator/software"):
        p = str(REPO_DIR / rel)
        if p not in sys.path:
            sys.path.insert(0, p)

    os.chdir(REPO_DIR / "emulator" / "notebooks" / "tutorial")

    # Verify right here, in this cell, instead of letting a failure surface
    # one cell later as a confusing ModuleNotFoundError.
    import importlib
    importlib.invalidate_caches()
    try:
        import qick
        print(f"\nColab setup complete. qick {qick.get_version()} importable from {qick.__file__}")
    except ImportError as e:
        raise RuntimeError(
            f"Setup finished but 'import qick' still fails ({e}). "
            f"cwd={pathlib.Path.cwd()}, REPO_DIR contents: {list(REPO_DIR.iterdir())}"
        ) from e
    print(f"Working directory: {pathlib.Path.cwd()}")
else:
    print("Not running in Colab -- skipping clone/build (using the local checkout as-is).")


## 1. Setup

In [2]:
# Jupyter notebook setup
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import sys
import pathlib
import logging

# Local qick_lib / emulator path setup (see emulator/notebooks/00_intro_emu.ipynb)
REPO_ROOT = pathlib.Path.cwd().parent.parent   # emulator/ (tutorial/ nests one level deeper)
sys.path.insert(0, str(REPO_ROOT / '..' / 'qick_lib'))
sys.path.insert(0, str(REPO_ROOT / 'software'))

import qick
from qick import *
from qick.asm_v2 import AveragerProgramV2, QickSweep1D, QickSpan

# Emulator entry point (drop-in replacement for QickSoc)
from qick_emu import QickEmu

logging.basicConfig(level=logging.INFO, format='%(levelname)-8s [%(filename)s:%(lineno)d] %(message)s')
logging.getLogger("qick_processor").setLevel(logging.WARNING)

# --- No firmware .bit file or physical board needed ---
# On real hardware:
#     BITSTREAM_PATH = '/path/to/your/firmware.bit'
#     soc = QickSoc(BITSTREAM_PATH)
CFG_PATH = REPO_ROOT / 'config' / 'qick_emu_config.json'
soc      = QickEmu(str(CFG_PATH))
soccfg   = soc.soccfg

# Define hardware channels (QICKEmu channel assignment, see 00_intro_emu.ipynb)
GEN_CH = 0
RO_CH  = 0

ARTIFACTS_ROOT = REPO_ROOT / 'artifacts' / '03_advanced_timing'
ARTIFACTS_ROOT.mkdir(parents=True, exist_ok=True)
print(f'Artifacts -> {ARTIFACTS_ROOT}')

WARNING  [qick_asm.py:49] QICK library version mismatch: 0.2.366 remote (the board), 0.2.88 local (the PC)
                        This may cause errors, usually KeyError in QickConfig initialization.
                        If this happens, you must bring your versions in sync.


Artifacts -> /home/larnaldi/git/emu_v1.2.0/emulator/artifacts/03_advanced_timing


## 2. Understanding the Timeline

In tProc v2, all timed instructions (pulses, triggers, delays) are scheduled on a common timeline. The tProc maintains a **reference time** that advances as instructions are executed.

**Key concepts:**
- **Absolute time:** The actual time from the start of the shot
- **Reference time:** The tProc's internal clock, which determines when the next instruction will execute
- **Scheduled time:** The time you specify for a pulse or trigger (e.g., `t=0.5`)

**The four timing commands:**

| Command | What it does | When to use |
| :--- | :--- | :--- |
| `delay(t)` | Adds `t` to the reference time | When you know exactly how long to wait |
| `wait(t)` | Waits until the reference time reaches `t` | When you need to wait for a specific absolute time |
| `delay_auto(t, gens=True, ros=True)` | Adds `t` PLUS the time needed to complete all pending gens/ROs | Safe way to ensure everything finishes |
| `wait_auto(t, gens=False, ros=True)` | Waits until reference time reaches `t` OR pending operations finish | Similar to v1's `sync_all()` |

## 3. Basic Timing: `delay()` vs `wait()`

The simplest timing commands are `delay()` and `wait()`. They work with absolute and relative timing.

In [3]:
class TimingExampleProgram(AveragerProgramV2):
    def _initialize(self, cfg):
        ro_ch = cfg['ro_ch']
        gen_ch = cfg['gen_ch']

        self.declare_gen(ch=gen_ch, nqz=1)
        self.declare_readout(ch=ro_ch, length=cfg['ro_len'])

        # Define a constant pulse
        self.add_pulse(ch=gen_ch, name="test_pulse",
                       style="const",
                       freq=cfg['freq'],
                       length=cfg['pulse_len'],
                       phase=0, gain=1.0)

        self.add_readoutconfig(ch=ro_ch, name="my_ro",
                               freq=cfg['freq'], gen_ch=gen_ch)
        self.send_readoutconfig(ch=ro_ch, name="my_ro", t=0)

    def _body(self, cfg):
        # Trigger readout at t=0.5 us
        self.trigger(ros=[cfg['ro_ch']], pins=[0], t=0.5)

        # First pulse at t=0
        self.pulse(ch=cfg['gen_ch'], name="test_pulse", t=0)

        if cfg['use_wait']:
            # Using wait: pause until reference time reaches 0.8 us
            self.wait(0.8)
        else:
            # Using delay: add 0.8 us to reference time
            self.delay(0.8)

        # Second pulse - when does it play?
        self.pulse(ch=cfg['gen_ch'], name="test_pulse", t=0)

# Compare delay vs wait
config = {
    'gen_ch': GEN_CH,
    'ro_ch': RO_CH,
    'freq': 100,
    'pulse_len': 0.1,
    'ro_len': 2.0,
    'use_wait': False
}

print("=== Using DELAY ===")
prog = TimingExampleProgram(soccfg, reps=1, final_delay=0.5, cfg=config)

# Get the compiled program as text to inspect the timing
print(prog.asm())

print("\n=== Using WAIT ===")
config['use_wait'] = True
prog = TimingExampleProgram(soccfg, reps=1, final_delay=0.5, cfg=config)
print(prog.asm())

WARNING  [asm_v2.py:998] warning: pulse time 0 appears to conflict with previous pulse ending at <qick.asm_v2.QickParam object at 0x780c80616b10>?


=== Using DELAY ===
     NOP 
     REG_WR s12 imm #0 
     WPORT_WR p4 wmem [&1] @0 
     TIME #430 inc_ref 
     REG_WR r0 imm #0 
reps:
     TRIG p0 set @215 
     TRIG p10 set @215 
     TRIG p0 clr @225 
     TRIG p10 clr @225 
     WPORT_WR p0 wmem [&0] @0 
     TIME #344 inc_ref 
     WPORT_WR p0 wmem [&0] @0 
     REG_WR s15 label SKIP 
     WAIT [s15] @731 time 
     TIME #946 inc_ref 
     REG_WR s12 op -op(s12 + #1) 
     REG_WR s15 label reps 
     TEST -op(r0 - #0) 
     JUMP [s15] -if(NZ) -wr(r0 op) -op(r0 + #1) 
     REG_WR s15 label NEXT 
     JUMP [s15] 


=== Using WAIT ===
     NOP 
     REG_WR s12 imm #0 
     WPORT_WR p4 wmem [&1] @0 
     TIME #430 inc_ref 
     REG_WR r0 imm #0 
reps:
     TRIG p0 set @215 
     TRIG p10 set @215 
     TRIG p0 clr @225 
     TRIG p10 clr @225 
     WPORT_WR p0 wmem [&0] @0 
     REG_WR s15 label SKIP 
     WAIT [s15] @344 time 
     WPORT_WR p0 wmem [&0] @0 
     REG_WR s15 label SKIP 
     WAIT [s15] @1075 time 
     TIME #1290 i

**Key difference:**
- `delay(0.8)` adds 0.8 us to the reference time, so the second pulse plays 0.8 us after the first pulse ends
- `wait(0.8)` pauses until the reference time reaches exactly 0.8 us (absolute time from shot start)

**Rule of thumb:** Use `delay()` for relative timing between pulses. Use `wait()` for absolute timing or when waiting for a specific absolute time.

## 4. Auto-Timing: `delay_auto()` and `wait_auto()`

The `_auto` variants automatically account for pending operations (pulses that haven't finished, readouts that are still acquiring). This is safer when you're not sure how long previous operations will take.

In [4]:
class AutoTimingProgram(AveragerProgramV2):
    def _initialize(self, cfg):
        ro_ch = cfg['ro_ch']
        gen_ch = cfg['gen_ch']

        self.declare_gen(ch=gen_ch, nqz=1)
        self.declare_readout(ch=ro_ch, length=cfg['ro_len'])

        # Define pulses with swept lengths
        self.add_pulse(ch=gen_ch, name="pulse_a",
                       style="const",
                       freq=cfg['freq'],
                       length=cfg['len_a'],
                       phase=0, gain=1.0)

        self.add_pulse(ch=gen_ch, name="pulse_b",
                       style="const",
                       freq=cfg['freq'],
                       length=cfg['len_b'],
                       phase=0, gain=1.0)

        self.add_readoutconfig(ch=ro_ch, name="my_ro",
                               freq=cfg['freq'], gen_ch=gen_ch)
        self.send_readoutconfig(ch=ro_ch, name="my_ro", t=0)

    def _body(self, cfg):
        # Play first pulse at t=0
        self.pulse(ch=cfg['gen_ch'], name="pulse_a", t=0)

        # Auto-delay: wait for pulse_a to finish, then add extra time
        # ros=False means don't wait for readout (which hasn't started yet)
        self.delay_auto(cfg['gap'], ros=False)

        # Play second pulse
        self.pulse(ch=cfg['gen_ch'], name="pulse_b", t=0)

        # Trigger readout after both pulses
        self.trigger(ros=[cfg['ro_ch']], pins=[0], t=cfg['trig_time'])

# Demo: pulse lengths vary, but the gap between them stays constant
config_auto = {
    'gen_ch': GEN_CH,
    'ro_ch': RO_CH,
    'freq': 100,
    'len_a': 0.2,      # us
    'len_b': 0.3,      # us (different length!)
    'gap': 0.1,        # us gap between pulses
    'trig_time': 0.8,  # us
    'ro_len': 1.5      # us
}

prog = AutoTimingProgram(soccfg, reps=1, final_delay=0.5, cfg=config_auto)

# Print the compiled program to see the generated assembly
print(prog.asm())

     NOP 
     REG_WR s12 imm #0 
     WPORT_WR p4 wmem [&2] @0 
     TIME #430 inc_ref 
     REG_WR r0 imm #0 
reps:
     WPORT_WR p0 wmem [&0] @0 
     TIME #129 inc_ref 
     WPORT_WR p0 wmem [&1] @0 
     TRIG p0 set @344 
     TRIG p10 set @344 
     TRIG p0 clr @354 
     TRIG p10 clr @354 
     REG_WR s15 label SKIP 
     WAIT [s15] @989 time 
     TIME #1205 inc_ref 
     REG_WR s12 op -op(s12 + #1) 
     REG_WR s15 label reps 
     TEST -op(r0 - #0) 
     JUMP [s15] -if(NZ) -wr(r0 op) -op(r0 + #1) 
     REG_WR s15 label NEXT 
     JUMP [s15] 



**Why `delay_auto()` is powerful:**
- It automatically calculates when pulse A ends
- Then schedules pulse B to start at `(end_of_pulse_A + gap)`
- This works even if pulse lengths change (e.g., in a sweep)

**Parameters:**
- `gens=True/False`: Whether to wait for generator operations (pulses)
- `ros=True/False`: Whether to wait for readout operations
- Default for `delay_auto()`: `gens=True, ros=True`
- Default for `wait_auto()`: `gens=False, ros=True` (like v1's `sync_all`)

## 5. Common Pitfall: Pulse Collisions

If you schedule a pulse before the previous one finishes, the tProc will give you a warning (if you have logging enabled) but will still execute. This can cause unexpected results.

In [5]:
class CollisionProgram(AveragerProgramV2):
    def _initialize(self, cfg):
        self.declare_gen(ch=cfg['gen_ch'], nqz=1)
        self.declare_readout(ch=cfg['ro_ch'], length=cfg['ro_len'])

        self.add_pulse(ch=cfg['gen_ch'], name="long_pulse",
                       style="const",
                       freq=cfg['freq'], length=0.5, phase=0, gain=1.0)

        self.add_readoutconfig(ch=cfg['ro_ch'], name="my_ro",
                               freq=cfg['freq'], gen_ch=cfg['gen_ch'])
        self.send_readoutconfig(ch=cfg['ro_ch'], name="my_ro", t=0)

    def _body(self, cfg):
        # First pulse at t=0
        self.pulse(ch=cfg['gen_ch'], name="long_pulse", t=0)

        # This pulse is scheduled while the first is still playing!
        # The tProc will issue a warning.
        self.pulse(ch=cfg['gen_ch'], name="long_pulse", t=0.3)

        self.trigger(ros=[cfg['ro_ch']], pins=[0], t=0.6)

config_collision = {
    'gen_ch': GEN_CH,
    'ro_ch': RO_CH,
    'freq': 100,
    'ro_len': 1.0
}

prog = CollisionProgram(soccfg, reps=1, final_delay=0.5, cfg=config_collision)
print("Program with potential pulse collision:")
print(prog.asm())
print("\nNote: The second pulse is scheduled before the first finishes!")

WARNING  [asm_v2.py:998] warning: pulse time 0.3 appears to conflict with previous pulse ending at <qick.asm_v2.QickParam object at 0x780c80632780>?


Program with potential pulse collision:
     NOP 
     REG_WR s12 imm #0 
     WPORT_WR p4 wmem [&1] @0 
     TIME #430 inc_ref 
     REG_WR r0 imm #0 
reps:
     WPORT_WR p0 wmem [&0] @0 
     WPORT_WR p0 wmem [&0] @129 
     TRIG p0 set @258 
     TRIG p10 set @258 
     TRIG p0 clr @268 
     TRIG p10 clr @268 
     REG_WR s15 label SKIP 
     WAIT [s15] @688 time 
     TIME #903 inc_ref 
     REG_WR s12 op -op(s12 + #1) 
     REG_WR s15 label reps 
     TEST -op(r0 - #0) 
     JUMP [s15] -if(NZ) -wr(r0 op) -op(r0 + #1) 
     REG_WR s15 label NEXT 
     JUMP [s15] 


Note: The second pulse is scheduled before the first finishes!


**How to fix:** Use `delay_auto()` to ensure proper spacing.

```python
# Instead of:
self.pulse(ch=gen_ch, name="long_pulse", t=0)
self.pulse(ch=gen_ch, name="long_pulse", t=0.3)  # Collision!

# Do this:
self.pulse(ch=gen_ch, name="long_pulse", t=0)
self.delay_auto(0.1)  # Wait for pulse to finish, then add 0.1 us
self.pulse(ch=gen_ch, name="long_pulse", t=0)  # Safely scheduled
```

## 6. Sweeping Delays

You can sweep delay times just like pulse parameters.

In [6]:
class DelaySweepProgram(AveragerProgramV2):
    def _initialize(self, cfg):
        self.declare_gen(ch=cfg['gen_ch'], nqz=1)
        self.declare_readout(ch=cfg['ro_ch'], length=cfg['ro_len'])

        self.add_loop("delay_sweep", self.cfg["steps"])

        self.add_pulse(ch=cfg['gen_ch'], name="test_pulse",
                       style="const",
                       freq=cfg['freq'], length=0.1,
                       phase=0, gain=1.0)

        self.add_readoutconfig(ch=cfg['ro_ch'], name="my_ro",
                               freq=cfg['freq'], gen_ch=cfg['gen_ch'])
        self.send_readoutconfig(ch=cfg['ro_ch'], name="my_ro", t=0)

    def _body(self, cfg):
        # Trigger readout at fixed time
        self.trigger(ros=[cfg['ro_ch']], pins=[0], t=0.5)

        # First pulse at t=0
        self.pulse(ch=cfg['gen_ch'], name="test_pulse", t=0)

        # Sweep the delay between pulses
        self.delay(QickSweep1D("delay_sweep", 0.1, 0.5))

        # Second pulse
        self.pulse(ch=cfg['gen_ch'], name="test_pulse", t=0)

config_delay_sweep = {
    'steps': 5,
    'gen_ch': GEN_CH,
    'ro_ch': RO_CH,
    'freq': 100,
    'ro_len': 1.0
}

prog = DelaySweepProgram(soccfg, reps=1, final_delay=0.5, cfg=config_delay_sweep)

# Show that the program compiles with swept delays
print("Program with swept delay:")
print(prog.asm())
print("\nNote: The tProc will automatically handle the varying delay times.")

Program with swept delay:
     NOP 
     REG_WR r2 imm #43 
     REG_WR r3 imm #817 
     REG_WR s12 imm #0 
     WPORT_WR p4 wmem [&1] @0 
     TIME #430 inc_ref 
     REG_WR r0 imm #0 
reps:
     REG_WR r1 imm #0 
delay_sweep:
     TRIG p0 set @215 
     TRIG p10 set @215 
     TRIG p0 clr @225 
     TRIG p10 clr @225 
     WPORT_WR p0 wmem [&0] @0 
     TIME inc_ref r2 
     WPORT_WR p0 wmem [&0] @0 
     REG_WR s15 label SKIP 
     WAIT [s15] @602 time 
     TIME inc_ref r3 
     REG_WR s12 op -op(s12 + #1) 
     REG_WR r2 op -op(r2 + #43) 
     REG_WR r3 op -op(r3 + #-43) 
     REG_WR s15 label delay_sweep 
     TEST -op(r1 - #4) 
     JUMP [s15] -if(NZ) -wr(r1 op) -op(r1 + #1) 
     REG_WR r2 op -op(r2 + #-215) 
     REG_WR r3 op -op(r3 + #215) 
     REG_WR s15 label reps 
     TEST -op(r0 - #0) 
     JUMP [s15] -if(NZ) -wr(r0 op) -op(r0 + #1) 
     REG_WR s15 label NEXT 
     JUMP [s15] 


Note: The tProc will automatically handle the varying delay times.


## 7. Summary

You have learned:
- The difference between `delay()` (relative) and `wait()` (absolute)
- How `delay_auto()` and `wait_auto()` automatically account for pending operations
- How to avoid pulse collisions by using proper timing commands
- How to sweep delay times
- How to inspect the generated assembly code using `asm()` to verify timing

**Best practices:**
1. **Use `delay_auto()` when you want to wait for pulses to finish** - It's safer and works with swept lengths
2. **Use `wait()` for absolute timing** - When you need to hit an exact absolute time
3. **Use `delay()` for simple relative timing** - When you know exactly how long to wait
4. **Always check for pulse collisions** - Use `print(prog.asm())` to inspect the generated code
5. **Use tags to identify instructions** - Tags help with debugging and retrieving parameters

**Next steps:** Proceed to [`04_Real_Time_Feedback_emu.ipynb`](./04_Real_Time_Feedback_emu.ipynb) to learn how to read measurement results and make decisions on the tProc in real-time.